# Opinion-Leader Origination Probability Probe

- **Project:** OLIM 2.0 opinion-leader case
- **Submodel ID:** SM-OL-ORIG-PROB-01
- **Probe type:** deterministic analytical mechanism probe
- **Status:** runnable

## Question and boundary

This notebook asks only how the proposed origination probability changes across rounds, agent roles, and parameter settings. It evaluates

$$
\pi_i^t = \operatorname{logit}^{-1}\!\left(\operatorname{logit}(\pi_0) - \lambda(t-1) + \beta_L L_i\right),
$$

where $\pi_0$ is the round-one probability for an ordinary agent, $\lambda$ is interest decay, $L_i$ indicates opinion-leader status, and $\beta_L$ is the leader log-odds advantage. The displayed leader multiplier is $\exp(\beta_L)$.

This is a provisional researcher-defined operationalization. It is not an empirical calibration. The probe excludes Bernoulli sampling, message creation, stance selection, diffusion, belief updating, and network adaptation.

## Expected behavior

- At round 1, an ordinary agent's probability equals $\pi_0$.
- Positive $\lambda$ makes origination probability decrease with round number.
- When the leader odds multiplier is 1, leaders and ordinary agents have equal probabilities.
- A multiplier above 1 increases leader odds by that factor at every round.
- All probabilities remain strictly between 0 and 1 for valid inputs.

In [ ]:
from math import log
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'opinion_model').exists():
            return candidate
    raise RuntimeError('Could not locate the opinion-model project root.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from opinion_model.opleader import origination_probability

ROUNDS = tuple(range(1, 51))
BASE_PROBABILITIES = (0.05, 0.10, 0.20)
INTEREST_DECAYS = (0.00, 0.03, 0.06)
LEADER_ODDS_MULTIPLIERS = (1.0, 2.0, 4.0)
REFERENCE = {
    'base_probability': 0.10,
    'interest_decay': 0.03,
    'leader_odds_multiplier': 4.0,
}


## Full parameter grid

The canonical package function is evaluated directly. The equation is not reimplemented in the notebook.

In [ ]:
records = []
for base_probability in BASE_PROBABILITIES:
    for decay in INTEREST_DECAYS:
        for odds_multiplier in LEADER_ODDS_MULTIPLIERS:
            for round_index in ROUNDS:
                for role, is_leader in (('ordinary', False), ('leader', True)):
                    records.append({
                        'round': round_index,
                        'role': role,
                        'base_probability': base_probability,
                        'interest_decay': decay,
                        'leader_odds_multiplier': odds_multiplier,
                        'origination_probability': origination_probability(
                            base_origination_probability=base_probability,
                            round_index=round_index,
                            is_leader=is_leader,
                            interest_decay=decay,
                            leader_log_odds_advantage=log(odds_multiplier),
                        ),
                    })

grid = pd.DataFrame.from_records(records)
grid.head(8)


## Deterministic analytical checks

In [ ]:
assert grid['origination_probability'].between(0.0, 1.0, inclusive='neither').all()

round_one_ordinary = grid.query("round == 1 and role == 'ordinary'")
assert np.allclose(
    round_one_ordinary['origination_probability'],
    round_one_ordinary['base_probability'],
)

for (base_probability, odds_multiplier, role), group in grid.groupby(
    ['base_probability', 'leader_odds_multiplier', 'role']
):
    zero_decay = group.query('interest_decay == 0.0').sort_values('round')
    assert np.allclose(
        zero_decay['origination_probability'],
        zero_decay['origination_probability'].iloc[0],
    )
    for decay in (0.03, 0.06):
        values = group.query('interest_decay == @decay').sort_values('round')[
            'origination_probability'
        ].to_numpy()
        assert np.all(np.diff(values) < 0.0)

for keys, group in grid.groupby(
    ['base_probability', 'interest_decay', 'leader_odds_multiplier', 'round']
):
    by_role = group.set_index('role')['origination_probability']
    ordinary_odds = by_role['ordinary'] / (1.0 - by_role['ordinary'])
    leader_odds = by_role['leader'] / (1.0 - by_role['leader'])
    assert np.isclose(leader_odds / ordinary_odds, keys[2])

print(f'All analytical checks passed for {len(grid):,} probability evaluations.')


## One-at-a-time sensitivity

Each panel varies one parameter while holding the other two at the reference values.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
role_styles = {'ordinary': '--', 'leader': '-'}

panel_specs = [
    ('base_probability', BASE_PROBABILITIES, r'$\pi_0$'),
    ('interest_decay', INTEREST_DECAYS, r'$\lambda$'),
    ('leader_odds_multiplier', LEADER_ODDS_MULTIPLIERS, r'$\exp(\beta_L)$'),
]

for axis, (parameter, values, label) in zip(axes, panel_specs):
    for value in values:
        conditions = {**REFERENCE, parameter: value}
        subset = grid[
            np.isclose(grid['base_probability'], conditions['base_probability'])
            & np.isclose(grid['interest_decay'], conditions['interest_decay'])
            & np.isclose(
                grid['leader_odds_multiplier'],
                conditions['leader_odds_multiplier'],
            )
        ]
        for role in ('ordinary', 'leader'):
            role_subset = subset.query('role == @role').sort_values('round')
            axis.plot(
                role_subset['round'],
                role_subset['origination_probability'],
                linestyle=role_styles[role],
                label=f'{label}={value:g}, {role}',
            )
    axis.set_title(f'Vary {label}')
    axis.set_xlabel('Round')
    axis.grid(alpha=0.25)

axes[0].set_ylabel('Origination probability')
for axis in axes:
    axis.legend(fontsize=8)
fig.suptitle('Opinion-leader origination probability: one-at-a-time sensitivity')
fig.tight_layout()
plt.show()


## Selected probabilities

The table provides exact values at four rounds for three illustrative configurations.

In [ ]:
SCENARIOS = (
    ('low, stable, equal roles', 0.05, 0.00, 1.0),
    ('reference', 0.10, 0.03, 4.0),
    ('high initial, fast decay, 2x leader odds', 0.20, 0.06, 2.0),
)
selected_records = []
for scenario, base_probability, decay, odds_multiplier in SCENARIOS:
    for round_index in (1, 10, 25, 50):
        for role, is_leader in (('ordinary', False), ('leader', True)):
            selected_records.append({
                'scenario': scenario,
                'round': round_index,
                'role': role,
                'probability': origination_probability(
                    base_origination_probability=base_probability,
                    round_index=round_index,
                    is_leader=is_leader,
                    interest_decay=decay,
                    leader_log_odds_advantage=log(odds_multiplier),
                ),
            })

selected = (
    pd.DataFrame(selected_records)
    .pivot(index=['scenario', 'round'], columns='role', values='probability')
    .reset_index()
)
selected[['scenario', 'round', 'ordinary', 'leader']].round(4)


## Result and disposition

The implementation satisfies the proposed function's structural expectations over the declared grid: the baseline fixes ordinary round-one probability, interest decay lowers both roles' probabilities, and the leader term maintains the specified odds multiplier. These results establish mathematical and implementation consistency only. They do not identify empirically appropriate parameter values or validate the mechanism as a description of observed behavior.

The next modeling decision is therefore parameter justification or calibration; stochastic origination outcomes and downstream message flow remain outside this probe.